<div style="background: #f48fb1; padding: 30px; border-radius: 12px; text-align: center; margin-bottom: 10px;">
  <h1 style="color: white; font-size: 2.4em; margin: 0;">Equalizador Digital de Áudio</h1>
  <h3 style="color: white; margin: 8px 0 0 0; font-weight: 300;">Projeto 7 — Filtros Digitais</h3>
  <hr style="border: 1px solid white; margin: 16px auto; width: 60%; opacity: 0.5;">
  <p style="color: white; font-size: 1em; margin: 0;">Camada Física da Computação &nbsp;|&nbsp; Engenharia da Computação &nbsp;|&nbsp; Prof. Rodrigo Carareto</p>
  <p style="color: white; font-size: 1em; margin: 0;">Heloísa e Kailyn</p>
</div>

---

## Sobre o Projeto

Um **filtro digital** é um sistema que processa sinais digitais com o objetivo de modificar ou extrair certas características do sinal. Ele atua como uma *peneira de frequências*: deixa passar apenas as frequências desejadas e bloqueia (ou amplifica) as demais.

Neste projeto, implementamos um **equalizador gráfico de áudio de 12 bandas**, similar ao que você encontra em aplicativos de música como Spotify ou VLC. O usuário pode configurar o ganho de cada banda de frequência entre **−10 dB e +10 dB**.

### Bandas de frequência

| Banda | Frequência Central | Região |
|-------|--------------------|--------|
| 1 | 32 Hz | Sub-grave |
| 2 | 64 Hz | Grave |
| 3 | 125 Hz | Grave médio |
| 4 | 250 Hz | Médio |
| 5 | 500 Hz | Médio |
| 6 | 1.000 Hz | Médio agudo |
| 7 | 2.000 Hz | Agudo |
| 8 | 4.000 Hz | Agudo |
| 9 | 8.000 Hz | Presença |
| 10 | 16.000 Hz | Brilho |
| 11 | 18.000 Hz | Ultra-agudo  |
| 12 | 20.000 Hz | Ultra-agudo  |

#### Como o projeto funciona?

1. Carregamos um áudio gravado como uma lista de números – cada número é uma amostra da amplitude do sinal num instante de tempo. A taxa de amostragem é $44100 Hz$, ou seja, $44100$ amostras por segundo. O sinal é normalizado para o intervalo $[−1,1][-1, 1]$
$[−1,1]$ para evitar overflow ao aplicar os filtros;

2. O usuário configura os ganhos através de 12 sliders interativos, um para cada banda de frequência (de $32Hz$ a $20 kHz$);

3. Para cada banda, calculamos os coeficientes do filtro Peaking EQ, um filtro biquad de 2ª ordem cuja função de transferência é: $$G(Z) = \frac{b_0 + b_1 Z^{-1} + b_2 Z^{-2}}{a_0 + a_1 Z^{-1} + a_2 Z^{-2}}$$
Os coeficientes dependem de três parâmetros: a frequência central $f_0$, o ganho em $dB$ e o fator de qualidade $Q$;

4. Os 12 filtros são aplicados em cascata – a saída de um vira a entrada do próximo, o que equivale a, matemticamente, multiplicar as funções de transferência. No código, isso é feito com `lfilter()`, que aplica a equação de recorrência amostra por amostra:
$$Y[k] = b_0 X[k] + b_1 X[k-1] + b_2 X[k-2] - a_1 Y[k-1] - a_2 Y[k-2]$$

5. Plotamos a FFT do áudio original e do filtrado para visualizar as mudanças no espectro de frequências;

6. Reproduzimos os dois áudios em sequência para o usuário perceber a diferença auditivamente.

7. O Diagrama de Bode do filtro completo é gerado multiplicando as respostas em frequência $H(w)$ individuais de cada banda – o que é equivalente a multiplicar as funções de transferência, já que `freqz` avalia G(z) ponto a ponto no espectro.


#### Por que Peaking EQ e não outro filtro?
O Peaking EQ amplifica ou atenua ao redor de uma frequência central e deixa o resto do espectro intacto. Quando o ganho é 0$db$, o filtro é completajmente transparente: $G(Z)=1$

#### O que é o parâmetro Q?
$Q = f_0 / BW$ é a razão entre a frequência central e a largura de banda afetada. 
> $Q = 1.4$ fixo é o valor teórico padrão para equalizadores gráficos com bandas separadas por oitava (dobro), garantindo que as bandas se encaixem sem se sobrepor excessivamente. O usuário controla apenas o ganho de cada banda e o Q é uma constante projetada para que elas se encaixem bem sem deixar "buracos" ou se sobrepor demais. O Q controla a seletividade do filtro.

> Quanto maior o Q, menor a largura de banda BW e mais seletivo o filtro. A frequência central não se desloca com Q — apenas a largura da faixa passante muda. Nas bandas de rejeição, a atenuação aumenta conforme Q cresce. Cada banda afeta uma faixa de largura definida pelo Q, e as bandas adjacentes se somam criando esse efeito ondulado.
- Q baixo → banda larga → afeta muitas frequências vizinhas
- Q alto → banda estreita → afeta só aquela frequência específica

---

### 1. Imports e Configurações

In [ ]:
import numpy as np
from scipy.signal import iirpeak, freqz, TransferFunction, lfilter
import matplotlib.pyplot as plt
import sounddevice as sd
from scipy.io import wavfile
import ipywidgets as widgets
from IPython.display import display, Audio
import time 

---

### 2. Carregamento do Áudio

> O sinal de áudio eh uma sequência de amostras $X[k]$, captadas a uma taxa de amostragem $f_s = 44100$ Hz. Cada valor representa a  amplitude do sinal num instante discreto de tempo.

> **A equação central:** Cada sample de saída $Y[k]$ é uma combinação ponderada de samples anteriores de saída **e** de entrada:
>$$Y[k] = b_0 \cdot X[k] + b_1 \cdot X[k-1] + b_2 \cdot X[k-2] - a_1 \cdot Y[k-1] - a_2 \cdot Y[k-2]$$
>Os coeficientes $b$ (numerador) e $a$ (denominador) **definem completamente o filtro**.

Gravação do áudio

In [ ]:
# duracao = 15 # segundos
# fs = 44100 # freq de amostragem

# for i in range(3, 0, -1):
#     print(f"Gravando em {i}...", end='\r', flush=True)
#     time.sleep(1)

# print("Gravando...          ")  # espaços pra limpar o texto anterior
# audio_gravado = sd.rec(int(duracao * fs), samplerate=fs, channels=1, dtype='float64')
# sd.wait()
# print("Gravação concluída! Salvo como audio.wav")

# wavfile.write("audio.wav", fs, audio_gravado)


Áudio pronto (sem gravar)

In [ ]:
# Carrega o arquivo de áudio (.wav)
# wavfile.read retorna a taxa de amostragem fs e o sinal X[k] como array numpy
fs, audio = wavfile.read("musica.wav")

# Se o áudio for estéreo (2 canais), converte para mono tirando a média dos canais.
# Isso simplifica o processamento — o filtro será aplicado em um único sinal 1D.
if audio.ndim == 2:
    audio = audio.mean(axis=1)

# Normaliza o sinal para o intervalo [-1.0, 1.0]
# Isso é importante para evitar overflow numérico ao aplicar os filtros
audio = audio.astype(np.float64)
# wavfile carrega o áudio como inteiro (int16 geralmente), 
# e a divisão pode dar problema sem converter antes para float.
audio = audio / np.max(np.abs(audio))

# Recorta o áudio para 15 segundos a partir do segundo inicial desejado
inicio = 0   # segundo de início do trecho
duracao = 15  # duração em segundos
audio = audio[int(inicio * fs) : int((inicio + duracao) * fs)]

print(f"Taxa de amostragem: {fs} Hz")
print(f"Duração: {len(audio)/fs:.2f} segundos")
print(f"Total de amostras: {len(audio)}")

# Reproduz o áudio original para referência
# print("\nReproduzindo áudio original...")
# sd.play(audio, fs)
# sd.wait()

---

### 3. Função do Filtro Peaking EQ

> O filtro ***Peaking EQ*** é o coração do equalizador. Para cada banda, ele gera uma função de transferência $G(Z)$ que amplifica ou atenua frequências próximas à uma frequência central $f0$, com intensidade definida pelo ganho em dB e largura de banda controlada pelo fator de qualidade $Q$. 

>O filtro Peaking EQ gera uma função de transferência $G(Z)$ do tipo **biquad (2ª ordem)**:

$$G(Z) = \frac{b_0 + b_1 Z^{-1} + b_2 Z^{-2}}{a_0 + a_1 Z^{-1} + a_2 Z^{-2}}$$

Os coeficientes $b$ (numerador) e $a$ (denominador) são calculados a partir da frequência central $f_0$, do ganho em dB e do fator de qualidade $Q$.

In [ ]:
def peaking_eq(f0, gain_db, Q, fs):
    """
    Calcula os coeficientes do filtro Peaking EQ.

    Parâmetros:
        f0       : frequência central da banda (Hz)
        gain_db  : ganho desejado em dB (positivo = amplifica, negativo = atenua)
        Q        : fator de qualidade — controla a largura da banda afetada
        fs       : taxa de amostragem (Hz)

    Retorna:
        b, a : coeficientes do numerador e denominador da função de transferência G(Z)
    """
    A = 10 ** (gain_db / 40)          # Amplitude linear correspondente ao ganho em dB
    omega = 2 * np.pi * f0 / fs       # Frequência angular normalizada
    alpha = np.sin(omega) / (2 * Q)   # Parâmetro de largura de banda

    # Coeficientes do numerador (b) e denominador (a)
    b0 =  1 + alpha * A
    b1 = -2 * np.cos(omega)
    b2 =  1 - alpha * A
    a0 =  1 + alpha / A
    a1 = -2 * np.cos(omega)
    a2 =  1 - alpha / A

    # Normaliza pelo a0 para que o denominador comece em 1
    b = np.array([b0, b1, b2]) / a0
    a = np.array([a0, a1, a2]) / a0

    return b, a


# Teste rápido: filtro centrado em 1000 Hz com ganho de +6 dB
b_teste, a_teste = peaking_eq(f0=1000, gain_db=6, Q=1.0, fs=fs)

w, h = freqz(b_teste, a_teste, fs=fs)
plt.figure(figsize=(8, 4))
plt.semilogx(w, 20 * np.log10(abs(h)))
plt.title("Peaking EQ — Teste: 1000 Hz, +6 dB")
plt.xlabel("Frequência [Hz]")
plt.ylabel("Ganho [dB]")
plt.ylim(-15, 15)
plt.grid(which='both', linestyle='--', linewidth=0.5)
plt.axvline(1000, color='red', linestyle=':', label='Frequência central')
plt.legend()
plt.tight_layout()
plt.show()

Plotando o gráfico do teste, vemos que:
- O pico esta em 1000Hz, que eh a freq central $f0$;
- O ganho nop pico eh $+6dB$, que eh oq configuramos;
- Fora da banda, o ganho eh $0dB$, então o filtro não afeta as outras frequências

---

### 4. Interface com o Usuário (Sliders)

> O equalizador possui 12 bandas de frequência. Para cada banda, o usuário configura o ganho entre $-10dB$ e $+10dB$. O fator de qualidade Q = 1.4 é um valor padrão comum em equalizadores gráficos de 12 bandas

In [ ]:
# Frequências centrais seguindo a ISO 266 e o enunciado do projeto (imagem do equalizador)
BANDAS = [32, 64, 125, 250, 500, 1000, 2000, 4000, 8000, 16000, 18000, 20000]
# As últimas duas bandas (18 kHz e 20 kHz) completam as 12 do enunciado 
# — o enunciado mostrou até 16 kHz na imagem mas pediu 12 bandas, então fechamos com 18k e 20k que são frequências audiáveis reais

# Q = 1/( sqrt(2) - 1/sqrt(2) ) ≈ 1.41 — valor teórico para equalizador gráfico de oitava
Q = 1.4

# Cria um slider para cada banda
sliders = []
for f in BANDAS:
    label = f"{f} Hz" if f < 1000 else f"{f//1000} kHz"
    slider = widgets.FloatSlider(
        value=0,
        min=-10,
        max=10,
        step=0.5,
        description=label,
        style={'description_width': '60px'},
        layout=widgets.Layout(width='400px')
    )
    sliders.append(slider)

# Botão para aplicar o equalizador
botao = widgets.Button(
    description="Aplicar Equalizador",
    button_style='primary',
    layout=widgets.Layout(width='200px', margin='15px 0px 0px 0px')
)

# Organiza os sliders em 2 colunas
coluna1 = widgets.VBox(sliders[:6])
coluna2 = widgets.VBox(sliders[6:])
painel = widgets.HBox([coluna1, coluna2])

# Exibe a interface
display(widgets.VBox([painel, botao]))

> ### Como usar o equalizador
> O equalizador possui **12 bandas de frequência**, cada uma controlada por um slider.
Cada slider representa uma frequência central e permite configurar o **ganho** daquela banda:
> - Mover o slider para **cima (direita)** → **amplifica** aquela faixa de frequência (até +10 dB)
> - Mover o slider para **baixo (esquerda)** → **atenua** aquela faixa de frequência (até -10 dB)
> - Slider no **centro (0 dB)** → aquela banda não é alterada
> Após configurar os sliders, clique em **"Aplicar Equalizador"** para processar o áudio.
>O programa irá então aplicar os 12 filtros Peaking EQ em cascata — ou seja, a saída $Y[k]$ de um filtro se torna a entrada $X[k]$ do próximo — e reproduzir o áudio filtrado logo em seguida.

---

### 5. Aplicação dos Filtros em Cascata

> Aqui, conectamos o botão "Aplicar Equalizador" à função que processa o áudio.
> Os 12 filtros Peaking EQ são aplicados em cascata: a saída $Y[k]$ de um filtro se torna a entrada $X[k]$ do próximo. Matematicamente, isso equivale a multiplicar as funções de transferência de cada banda:
>  $$G_{total}(Z) = G_1(Z) \cdot G_2(Z) \cdot \ldots \cdot G_{12}(Z)$$

> A função `lfilter(b, a, sinal)` aplica a equação de recorrência do filtro diretamente sobre o sinal discreto $X[k]$, produzindo $Y[k]$. 

---

### 6. Reprodução do Áudio

> Além de reproduzir os áudios, vamos plotar a FFT do sinal original e do filtrado. A FFT decompõe o sinal no domínio da frequência, mostrando quais frequências estão presentes e com qual amplitude. Isso permite verificar visualmente o efeito do equalizador.

In [ ]:
def aplicar_e_visualizar(b):
    # Lê o ganho de cada slider
    ganhos = [slider.value for slider in sliders]

    # Aplica os filtros em cascata
    audio_filtrado = audio.copy()
    for f0, gain_db in zip(BANDAS, ganhos):
        if gain_db != 0:
            b_coef, a_coef = peaking_eq(f0, gain_db, Q, fs)
            audio_filtrado = lfilter(b_coef, a_coef, audio_filtrado)

    # Normaliza só se clipar
    max_val = np.max(np.abs(audio_filtrado))
    if max_val > 1.0:
        audio_filtrado = audio_filtrado / max_val

    # FFT do sinal original e filtrado 
    N = len(audio)
    freqs = np.fft.rfftfreq(N, d=1/fs)         # Eixo de frequências
    fft_original = np.abs(np.fft.rfft(audio))   # Espectro do sinal original
    fft_filtrado = np.abs(np.fft.rfft(audio_filtrado))  # Espectro do filtrado

    fft_original_db = 20 * np.log10(fft_original[1:] + 1e-10)
    fft_filtrado_db = 20 * np.log10(fft_filtrado[1:] + 1e-10)

    # Ticks do eixo X legíveis
    ticks = [20, 50, 100, 200, 500, 1000, 2000, 5000, 10000, 20000]
    tick_labels = ['20', '50', '100', '200', '500', '1k', '2k', '5k', '10k', '20k']

    # Plota as FFTs
    fig, axs = plt.subplots(3, 1, figsize=(10, 9))

    # Gráfico 1: sobreposição
    axs[0].semilogx(freqs[1:], fft_original_db, color='#c2006e', label='Original', alpha=0.9, linewidth=1.2)
    axs[0].semilogx(freqs[1:], fft_filtrado_db, color='#ff9de2', label='Filtrado', alpha=0.9, linewidth=1.2)
    axs[0].set_title("Comparação: Original vs Filtrado")
    axs[0].set_ylabel("Magnitude [dB]")
    axs[0].set_xlabel("Frequência [Hz]")
    axs[0].set_xticks(ticks)
    axs[0].set_xticklabels(tick_labels)
    axs[0].set_xlim(20, 20000)
    axs[0].legend()
    axs[0].grid(which='both', linestyle='--', linewidth=0.5)

    # Gráfico 2: diferença aplicada pelo EQ, limitada ao range dos sliders (-10 a +10 dB)
    diferenca = fft_filtrado_db - fft_original_db
    axs[1].semilogx(freqs[1:], diferenca, color='#7b2d8b', linewidth=1.2)
    axs[1].axhline(0, color='gray', linewidth=0.8, linestyle='--')
    axs[1].fill_between(freqs[1:], diferenca, 0,
                         where=(diferenca > 0), color='#7b2d8b', alpha=0.3, label='boost')
    axs[1].fill_between(freqs[1:], diferenca, 0,
                         where=(diferenca < 0), color='#d62839', alpha=0.3, label='corte')
    axs[1].set_title("Diferença aplicada pelo EQ")
    axs[1].set_ylabel("Ganho [dB]")
    axs[1].set_xlabel("Frequência [Hz]")
    axs[1].set_xticks(ticks)
    axs[1].set_xticklabels(tick_labels)
    axs[1].set_xlim(20, 20000)
    axs[1].set_ylim(-10, 10)
    axs[1].legend()
    axs[1].grid(which='both', linestyle='--', linewidth=0.5)

    # Gráfico 3: zoom na região de voz (100Hz - 8kHz)
    mask = (freqs[1:] >= 100) & (freqs[1:] <= 8000)
    ticks_zoom = [100, 200, 500, 1000, 2000, 5000, 8000]
    labels_zoom = ['100', '200', '500', '1k', '2k', '5k', '8k']
    axs[2].semilogx(freqs[1:][mask], fft_original_db[mask], color='#c2006e', label='Original', alpha=0.9, linewidth=1.2)
    axs[2].semilogx(freqs[1:][mask], fft_filtrado_db[mask], color='#ff9de2', label='Filtrado', alpha=0.9, linewidth=1.2)
    axs[2].set_title("Zoom: 100 Hz – 8 kHz")
    axs[2].set_ylabel("Magnitude [dB]")
    axs[2].set_xlabel("Frequência [Hz]")
    axs[2].set_xticks(ticks_zoom)
    axs[2].set_xticklabels(labels_zoom)
    axs[2].set_xlim(100, 8000)
    axs[2].legend()
    axs[2].grid(which='both', linestyle='--', linewidth=0.5)

    plt.tight_layout()
    plt.show()

    # Reprodução dos áudios
    print("Reproduzindo áudio ORIGINAL...")
    sd.play(audio, fs)
    sd.wait()

    print("Reproduzindo áudio FILTRADO...")
    sd.play(audio_filtrado, fs)
    sd.wait()
    print("Concluído!")

# Reconecta o botão à nova função (com visualização)
botao.on_click(aplicar_e_visualizar)
print("Pronto! Configure os sliders e clique em 'Aplicar Equalizador'.")

---

### 7. Diagrama de Bode do Filtro Completo

> O diagrama de Bode do filtro completo mostra o efeito combinado de todos os 12 filtros ***Peaking EQ*** aplicados em cascata.
> Para obter a função de transferência equivalente, multiplicamos as funções de transferência individuais de cada banda:
> $$G_{total}(Z) = G_1(Z) \cdot G_2(Z) \cdot \ldots \cdot G_{12}(Z)$$

> Para cada banda, calculamos $H(w)$ via `freqz()` — que avalia a função de transferência $G(Z)$ substituindo $Z = e^{j\omega}$ para cada frequência. Depois multiplicamos todos os $H$ complexos entre si. O módulo do produto em dB nos dá a curva de Bode do equalizador inteiro, mostrando exatamente quanto cada frequência está sendo amplificada ou atenuada pela configuração atual dos sliders.

> O diagrama Mostra a resposta em frequência do filtro completo com as 12 bandas combinadas. É o que o equalizador faz em cada frequência independente do áudio.
> Os "ondulados" na curva são os picos/vales individuais de cada banda se somando — dá pra contar aproximadamente 12 ondulações, uma por banda


In [ ]:
def plotar_bode(b):
    ganhos = [slider.value for slider in sliders]

    # Calcula a resposta em frequência total multiplicando as respostas individuais
    w, _ = freqz([1], [1], worN=8192, fs=fs)  # Define o eixo de frequências
    H_total = np.ones(len(w), dtype=complex)   # Começa com ganho 1 (neutro)

    for f0, gain_db in zip(BANDAS, ganhos):
        b_coef, a_coef = peaking_eq(f0, gain_db, Q, fs)
        _, H = freqz(b_coef, a_coef, worN=8192, fs=fs)
        H_total *= H  # Multiplica as funções de transferência em cascata

    # Plota o diagrama de Bode
    plt.figure(figsize=(10, 5))
    plt.semilogx(w, 20 * np.log10(np.abs(H_total)), color='hotpink', linewidth=2)
    plt.title("Diagrama de Bode — Filtro Completo (12 bandas)")
    plt.xlabel("Frequência [Hz]")
    plt.ylabel("Ganho [dB]")
    plt.ylim(-15, 15)
    plt.grid(which='both', linestyle='--', linewidth=0.5)

    # Marca as frequências centrais de cada banda
    for f0, gain_db in zip(BANDAS, ganhos):
        label = f"{f0} Hz" if f0 < 1000 else f"{f0//1000} kHz"
        plt.axvline(f0, color='gray', linestyle=':', linewidth=0.8)
        plt.annotate(label, xy=(f0, gain_db),
                     fontsize=7, ha='center', color='gray',
                     xytext=(0, 5), textcoords='offset points')

    plt.tight_layout()
    plt.show()

# Cria um botão separado para o Bode
botao_bode = widgets.Button(
    description="Plotar Diagrama de Bode",
    button_style='warning',
    layout=widgets.Layout(width='220px')
)
botao_bode._click_handlers.callbacks.clear()
botao_bode.on_click(plotar_bode)
display(botao_bode)
print("Configure os sliders na seção 4 e clique para ver o Bode do filtro completo.")